# PathSnap - Model: XGBoost (gradient-boosted trees)

**Family:** Boosting. **Task framing:** each GPS observation has a set of candidate road arcs; the model scores every candidate (`label = 1` if it is the reference arc). The highest-scoring candidate is the observation-level prediction, and the probabilities become emission scores for the HMM/Viterbi stage.

Same data, features, group-aware CV and metrics as the other PathSnap model notebooks, so results are directly comparable.

**Run order:** top to bottom. Outputs go to `results/xgboost/`.

## 0. Setup

In [ ]:
!pip install -q xgboost  # no-op if already installed

In [ ]:
# ---- Paths: Colab (Drive zip) or local run --------------------------------------------
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path("/content/drive/MyDrive/pathsnap_features.zip")
    DATA = Path("/content/pathsnap_features")
    OUT = Path("/content/drive/MyDrive/PathSnap/results/xgboost")
    if not (DATA / "data_features").exists():
        with zipfile.ZipFile(ZIP) as z:
            z.extractall(DATA)
else:
    DATA = Path(os.environ.get("PATHSNAP_DATA", "../.."))
    OUT = Path(os.environ.get("PATHSNAP_OUT", "results/xgboost"))

FEAT_DIR = DATA / "data_features"
FLAGS = DATA / "trip_flags.csv" if (DATA / "trip_flags.csv").exists() else DATA / "data_clean" / "trip_flags.csv"
OUT.mkdir(parents=True, exist_ok=True)
print("features:", FEAT_DIR, "| flags:", FLAGS, "| output:", OUT)

In [ ]:
import json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib, xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, ConfusionMatrixDisplay)

SEED = 42
np.random.seed(SEED)

FEATURES = json.loads((FEAT_DIR / "feature_params.json").read_text())["FEATURES"]

CONFIG = {
    "model": "XGBClassifier",
    "family": "Boosting",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_estimators": 300,
    "max_depth": 10,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "reg_lambda": 1.0,
    "scale_pos_weight": 1.0,   # unweighted keeps probabilities calibrated for the HMM
    "scaler": "none (tree model, scale-invariant)",
    "early_stopping": "none (fixed n_estimators; no selection on validation data)",
    "cv": "GroupKFold(n_splits=5) on trip_id",
    "seed": SEED,
    "n_features": len(FEATURES),
}

def make_model(cfg=CONFIG):
    return XGBClassifier(
        objective=cfg["objective"], eval_metric=cfg["eval_metric"], tree_method=cfg["tree_method"],
        n_estimators=cfg["n_estimators"], max_depth=cfg["max_depth"], learning_rate=cfg["learning_rate"],
        subsample=cfg["subsample"], colsample_bytree=cfg["colsample_bytree"],
        min_child_weight=cfg["min_child_weight"], reg_lambda=cfg["reg_lambda"],
        scale_pos_weight=cfg["scale_pos_weight"], n_jobs=-1, random_state=cfg["seed"],
    )

print(f"xgboost {xgboost.__version__} | sklearn {sklearn.__version__} | {len(FEATURES)} features")

## 1. Load data

* **Train:** only trips with `use_for_tuning = True` (trips 02, 33, 43, 54 have partial reference routes and are excluded).
* Rows with `label = -1` (outside the route's label span, or alignment failures) are dropped.
* Per-observation reference info (`*_points.csv.gz`) is loaded separately: it is needed to count observations whose reference arc has **no candidate row at all**, so top-1 accuracy is not inflated.

In [ ]:
flags = pd.read_csv(FLAGS, dtype={"trip_id": str})

def trips_of(part):
    t = flags[flags.partition == part]
    if part == "train":
        t = t[t.use_for_tuning.astype(bool)]
    return sorted(t.trip_id)

def load_partition(part):
    rows, pts = [], []
    for tid in trips_of(part):
        f = pd.read_csv(FEAT_DIR / part / f"{tid}.csv.gz", dtype={"trip_id": str})
        y = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}.csv.gz")
        assert len(f) == len(y) and (f.idx.values == y.idx.values).all() and (f.arc_id.values == y.arc_id.values).all()
        f["label"] = y.label.values
        rows.append(f[f.label >= 0])
        p = pd.read_csv(FEAT_DIR / "labels" / part / f"{tid}_points.csv.gz")
        p.insert(0, "trip_id", tid)
        pts.append(p[p.labelled])
    return pd.concat(rows, ignore_index=True), pd.concat(pts, ignore_index=True)

t0 = time.time()
train, train_pts = load_partition("train")
test, test_pts = load_partition("student_test")
print(f"loaded in {time.time() - t0:.1f}s")
for name, d, p in [("train", train, train_pts), ("student_test", test, test_pts)]:
    print(f"{name:13} trips {d.trip_id.nunique():3} | rows {len(d):>9,} | positive rate {d.label.mean():.3f} "
          f"| labelled observations {len(p):>7,} | candidate miss rate {1 - p.ref_in_candidates.mean():.4%}")

## 2. Metrics

**Row level** (threshold 0.5): accuracy, precision, recall, F1, confusion matrix, log loss. Only ~14% of rows are positive, so row accuracy is misleading on its own.

**Observation level** (the primary metric):
* **Top-1 accuracy** — for each labelled GPS observation, take the candidate with the highest score; correct if it is the reference arc. Observations with no correct candidate count as **wrong**.
* **Candidate miss rate** — share of labelled observations whose reference arc is not in the candidate set (an upper bound on achievable top-1). It does not depend on the model.

The nearest-arc rule (`dist_rank == 0`) is reported alongside as a non-ML reference.

In [ ]:
def row_metrics(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat, zero_division=0),
        "recall": recall_score(y, yhat, zero_division=0),
        "f1": f1_score(y, yhat, zero_division=0),
        "log_loss": log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]),
    }

def top1(df, score, pts):
    # top-1 over ALL labelled observations of the trips in df (no candidate = wrong)
    d = df[["trip_id", "idx", "label"]].assign(s=score)
    best = d.loc[d.groupby(["trip_id", "idx"]).s.idxmax()]
    hits = int(best.label.sum())
    n_obs = len(pts[pts.trip_id.isin(df.trip_id.unique())])
    n_reachable = int(pts[pts.trip_id.isin(df.trip_id.unique())].ref_in_candidates.sum())
    return {"top1": hits / n_obs, "top1_given_candidate": hits / max(n_reachable, 1), "n_obs": n_obs}

def evaluate(df, p, pts):
    m = row_metrics(df.label.values, p)
    m.update(top1(df, p, pts))
    nearest = top1(df, -df.dist_rank.values.astype(float), pts)
    m["top1_nearest_arc"] = nearest["top1"]
    return m

## 3. Cross-validation — GroupKFold by trip (train only)

All rows of a trip stay in the same fold: rows from one trip are strongly correlated, so a random row split would overstate accuracy. XGBoost needs no scaling.

The per-round train/validation log loss of each fold is **recorded for diagnosis only** (to judge over/underfitting). It is not used for early stopping or any selection.

In [ ]:
gkf = GroupKFold(n_splits=5)
cv_rows, cv_curves = [], []
for fold, (tr, va) in enumerate(gkf.split(train, train.label, groups=train.trip_id)):
    a, b = train.iloc[tr], train.iloc[va]
    model = make_model()
    t0 = time.time()
    model.fit(a[FEATURES], a.label, eval_set=[(a[FEATURES], a.label), (b[FEATURES], b.label)], verbose=False)
    fit_s = time.time() - t0
    hist = model.evals_result()
    cv_curves.append({"train": hist["validation_0"]["logloss"], "val": hist["validation_1"]["logloss"]})
    t0 = time.time(); p = model.predict_proba(b[FEATURES])[:, 1]; pred_s = time.time() - t0
    m = evaluate(b, p, train_pts)
    m.update({"fold": fold, "train_trips": a.trip_id.nunique(), "val_trips": b.trip_id.nunique(),
              "fit_s": fit_s, "predict_s": pred_s, "predict_ms_per_1k_rows": 1000 * pred_s / len(b) * 1000})
    cv_rows.append(m)
    print(f"fold {fold}: top-1 {m['top1']:.4f} (nearest-arc {m['top1_nearest_arc']:.4f}) "
          f"| F1 {m['f1']:.4f} | log loss {m['log_loss']:.4f} | fit {fit_s:.1f}s")

cv = pd.DataFrame(cv_rows)
cols = ["top1", "top1_given_candidate", "top1_nearest_arc", "accuracy", "precision", "recall", "f1", "log_loss", "fit_s", "predict_s"]
cv_summary = cv[cols].agg(["mean", "std"]).T
cv_summary

## 4. Final model — fit on all usable train trips, evaluate on student_test

Student_test is used only to measure this finished model (development-stage comparison), never to choose hyperparameters. Its per-round log loss is logged for the learning curve only.

In [ ]:
final = make_model()
t0 = time.time()
final.fit(train[FEATURES], train.label,
          eval_set=[(train[FEATURES], train.label), (test[FEATURES], test.label)], verbose=False)
final_fit_s = time.time() - t0
final_hist = final.evals_result()
t0 = time.time(); p_test = final.predict_proba(test[FEATURES])[:, 1]; final_pred_s = time.time() - t0

test_metrics = evaluate(test, p_test, test_pts)
test_metrics.update({"fit_s": final_fit_s, "predict_s": final_pred_s, "rows": len(test)})
pd.Series(test_metrics).round(4)

In [ ]:
# Per-trip top-1 on student_test
per_trip = []
for tid, g in test.assign(p=p_test).groupby("trip_id"):
    per_trip.append({"trip_id": tid, **top1(g, g.p.values, test_pts),
                     "top1_nearest_arc": top1(g, -g.dist_rank.values.astype(float), test_pts)["top1"]})
per_trip = pd.DataFrame(per_trip).sort_values("top1")
per_trip.round(4)

## 5. Diagnostics — learning curves, confusion matrix, feature importance

In [ ]:
# Learning curves from real per-round histories (CV folds + final model). Diagnostic only.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for k, c in enumerate(cv_curves):
    axes[0].plot(c["train"], color="#2a78d6", alpha=0.35, label="train (fold)" if k == 0 else None)
    axes[0].plot(c["val"], color="#eb6834", alpha=0.6, label="validation (fold)" if k == 0 else None)
axes[0].set_title("5-fold CV: log loss per boosting round")

axes[1].plot(final_hist["validation_0"]["logloss"], color="#2a78d6", label="train")
axes[1].plot(final_hist["validation_1"]["logloss"], color="#eb6834", label="student_test")
axes[1].set_title("Final model: log loss per boosting round")

for ax in axes:
    ax.set_xlabel("boosting round"); ax.set_ylabel("log loss"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "xgb_learning_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(test.label, (p_test >= 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["not ref (0)", "ref arc (1)"]).plot(ax=axes[0], colorbar=False, values_format=",")
axes[0].set_title("Student_test confusion matrix (rows, threshold 0.5)")

imp = pd.Series(final.feature_importances_, index=FEATURES).sort_values()
axes[1].barh(imp.index, imp.values, color="#2a78d6")
axes[1].set_title("Feature importance (gain)")
axes[1].tick_params(axis="y", labelsize=8)
fig.tight_layout()
fig.savefig(OUT / "xgb_confusion_and_importance.png", dpi=150)
plt.show()

## 6. Save model, experiment log and predictions

* `xgb_model.joblib` / `xgb_model.json` — the fitted model (sklearn wrapper / native format).
* `experiment_log.json` — configuration, seed, versions, per-fold CV results, student_test results and timings.
* `student_test_predictions.csv.gz` — per-candidate probabilities, to be used as emission scores in the HMM/Viterbi stage.

In [ ]:
joblib.dump(final, OUT / "xgb_model.joblib")
final.save_model(str(OUT / "xgb_model.json"))

log = {
    "config": CONFIG,
    "features": FEATURES,
    "versions": {"python": platform.python_version(), "xgboost": xgboost.__version__,
                 "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "data": {"train_trips": trips_of("train"), "student_test_trips": trips_of("student_test"),
             "train_rows": len(train), "student_test_rows": len(test)},
    "cv_folds": cv.to_dict(orient="records"),
    "cv_mean": cv[cols].mean().to_dict(),
    "cv_std": cv[cols].std().to_dict(),
    "cv_learning_curves": cv_curves,
    "final_learning_curve": {"train": final_hist["validation_0"]["logloss"],
                             "student_test": final_hist["validation_1"]["logloss"]},
    "student_test": test_metrics,
    "student_test_per_trip": per_trip.to_dict(orient="records"),
}
(OUT / "experiment_log.json").write_text(json.dumps(log, indent=2, default=float))

test[["trip_id", "idx", "arc_id", "label"]].assign(p_xgb=p_test).to_csv(
    OUT / "student_test_predictions.csv.gz", index=False, float_format="%.6f")
print("saved to", OUT)

## 7. Map — GPS vs predicted path vs reference route (student_test trip)

In [ ]:
# ---- Map: GPS vs predicted path vs reference route (run after the final-model cell) ----
import folium

TRIP = "00000077"

# raw coordinates come from the original dataset zip (features hold no coordinates)
RAW_ZIP = Path("/content/Student folder.zip")
RAW = Path("/content/raw") if IN_COLAB else DATA
if IN_COLAB and not (RAW / "Student folder").exists():
    with zipfile.ZipFile(RAW_ZIP) as z:
        z.extractall(RAW)
trip_dir = RAW / "Student folder" / "student_test" / TRIP

track = np.loadtxt(trip_dir / f"{TRIP}.track", ndmin=2)        # lon, lat, t   (row = idx)
nodes = np.loadtxt(trip_dir / f"{TRIP}.nodes", ndmin=2)        # lon, lat      (row = node id)
arcs = np.loadtxt(trip_dir / f"{TRIP}.arcs", dtype=int, ndmin=2)  # from, to   (row = arc id)
route = np.loadtxt(trip_dir / f"{TRIP}.route", dtype=int, ndmin=1)

# predicted arc per observation = highest-probability candidate
g = test[test.trip_id == TRIP].assign(p=p_test[(test.trip_id == TRIP).values])
best = g.loc[g.groupby("idx").p.idxmax(), ["idx", "arc_id", "label"]]
acc = best.label.mean()

def arc_line(a):  # folium wants [lat, lon]
    u, v = arcs[a]
    return [[nodes[u, 1], nodes[u, 0]], [nodes[v, 1], nodes[v, 0]]]

m = folium.Map(location=[track[:, 1].mean(), track[:, 0].mean()], zoom_start=14, tiles="OpenStreetMap")

ref = folium.FeatureGroup("Reference route (green)").add_to(m)
for a in route:
    folium.PolyLine(arc_line(a), color="green", weight=7, opacity=0.45).add_to(ref)

pred = folium.FeatureGroup("Predicted arcs (red)").add_to(m)
for a in best.arc_id.unique():
    folium.PolyLine(arc_line(a), color="red", weight=3, opacity=0.9).add_to(pred)

gps = folium.FeatureGroup("GPS points (blue = correct, orange = wrong)").add_to(m)
for i, ok in zip(best.idx, best.label):
    folium.CircleMarker([track[i, 1], track[i, 0]], radius=2, weight=0, fill=True, fill_opacity=0.8,
                        color="#2a78d6" if ok else "#eb6834", tooltip=f"idx {i}").add_to(gps)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[track[:, 1].min(), track[:, 0].min()], [track[:, 1].max(), track[:, 0].max()]])
print(f"Trip {TRIP}: {len(best):,} observations, top-1 {acc:.1%} | wrong points shown in orange")
m.save(str(OUT / f"map_{TRIP}.html"))
m